In [15]:
import os
import time
import glob
import xml.etree.ElementTree as ET
from pyspark.sql import SparkSession
from pyspark.sql.functions import (col, expr, rand, when, date_format, count,
                                    to_timestamp, concat, lit, least, greatest,
                                    mean, stddev, skewness, kurtosis, avg, broadcast)
from pyspark.sql.types import (IntegerType, StructType, StructField,
                                StringType, DoubleType)

# --- DYNAMIC PATH CONFIGURATION ---
# Automatically finds the 'official' folder whether run from root or notebooks folder
if "notebooks" in os.getcwd():
    BASE_DIR = os.path.dirname(os.getcwd())
else:
    BASE_DIR = os.getcwd()

TIMETABLE_XML_DIR = os.path.join(BASE_DIR, "data", "raw", "timetables")
FARES_XML_PATH = os.path.join(BASE_DIR, "data", "raw", "fares.xml")
OUTPUT_PARQUET = "file://" + os.path.join(BASE_DIR, "data", "processed", "operational_data.parquet")
DB_PATH = os.path.join(BASE_DIR, "db", "transport_analytics.db")

os.makedirs(os.path.join(BASE_DIR, "data", "processed"), exist_ok=True)
os.makedirs(os.path.join(BASE_DIR, "db"), exist_ok=True)

# --- SPARK SESSION ---
spark = SparkSession.builder \
    .appName("SmartCityTransitAnalytics_TransXChange") \
    .config("spark.sql.shuffle.partitions", "4") \
    .config("spark.driver.memory", "4g") \
    .config("spark.executor.memory", "4g") \
    .config("spark.sql.legacy.timeParserPolicy", "LEGACY") \
    .master("local[*]") \
    .getOrCreate()

print("SparkSession initialized!")

SparkSession initialized!


In [16]:
def _get_namespace(root):
    if root.tag.startswith('{'):
        return root.tag.split('}')[0] + '}'
    return ''

def parse_timetable_xml(xml_file_path):
    records = []
    try:
        tree = ET.parse(xml_file_path)
        root = tree.getroot()
        ns = _get_namespace(root)
        filename = os.path.basename(xml_file_path)

        jps_stop_counts = {}
        for jps in root.findall(f'.//{ns}JourneyPatternSection'):
            jps_id = jps.get('id', '')
            links = jps.findall(f'{ns}JourneyPatternTimingLink')
            jps_stop_counts[jps_id] = len(links)

        jp_stop_counts = {}
        for jp in root.findall(f'.//{ns}JourneyPattern'):
            jp_id = jp.get('id', '')
            total_stops = 0
            for ref in jp.findall(f'{ns}JourneyPatternSectionRef'):
                ref_id = ref.text.strip() if ref.text else ''
                total_stops += jps_stop_counts.get(ref_id, 0)
            if total_stops == 0:
                for ref in jp.findall(f'{ns}SectionRef'):
                    ref_id = ref.text.strip() if ref.text else ''
                    total_stops += jps_stop_counts.get(ref_id, 0)
            jp_stop_counts[jp_id] = max(total_stops, 1)

        service_lines = {}
        for service in root.findall(f'.//{ns}Service'):
            sc_elem = service.find(f'{ns}ServiceCode')
            service_code = (sc_elem.text.strip() if sc_elem is not None and sc_elem.text else 'UNKNOWN')
            line_name = 'UNKNOWN'
            ln_elem = service.find(f'.//{ns}LineName')
            if ln_elem is not None and ln_elem.text:
                line_name = ln_elem.text.strip()
            service_lines[service_code] = line_name

        for vj in root.findall(f'.//{ns}VehicleJourney'):
            vjc_elem = vj.find(f'{ns}VehicleJourneyCode')
            vj_code = (vjc_elem.text.strip() if vjc_elem is not None and vjc_elem.text else 'UNKNOWN')
            dt_elem = vj.find(f'{ns}DepartureTime')
            dep_time = (dt_elem.text.strip() if dt_elem is not None and dt_elem.text else None)
            jpr_elem = vj.find(f'{ns}JourneyPatternRef')
            jp_ref = (jpr_elem.text.strip() if jpr_elem is not None and jpr_elem.text else 'UNKNOWN')
            sc_elem = vj.find(f'{ns}ServiceRef')
            service_code = 'UNKNOWN'
            if sc_elem is not None and sc_elem.text:
                service_code = sc_elem.text.strip()
            line_name = service_lines.get(service_code, 'UNKNOWN')
            stop_count = jp_stop_counts.get(jp_ref, 1)

            if dep_time:
                records.append({
                    'SourceFile': filename, 'ServiceCode': service_code, 'LineName': line_name,
                    'VehicleJourneyCode': vj_code, 'JourneyPatternRef': jp_ref,
                    'DepartureTime': dep_time, 'Sequence': stop_count
                })
    except Exception as e:
        print(f"  Error parsing {xml_file_path}: {e}")
    return records

def parse_fares_xml(xml_file_path):
    records = []
    try:
        tree = ET.parse(xml_file_path)
        root = tree.getroot()
        def local_tag(elem):
            return elem.tag.split('}')[-1] if '}' in elem.tag else elem.tag
        found_fares = False
        for fare_frame in root.iter():
            if local_tag(fare_frame) == 'FareFrame':
                current_product = 'standard'
                for elem in fare_frame.iter():
                    tag = local_tag(elem)
                    if tag == 'Name' and elem.text:
                        text = elem.text.strip().lower()
                        if text in ['single', 'return', 'adult', 'child', 'day_ticket']:
                            current_product = text
                    if tag == 'Amount' and elem.text:
                        try:
                            price = float(elem.text.strip())
                            if price > 0:
                                records.append({'fare_class': 'adult', 'fare_price': price, 'fare_product': current_product})
                                found_fares = True
                        except (ValueError, TypeError):
                            continue
        if not found_fares:
            for elem in root.iter():
                tag = local_tag(elem).lower()
                if tag in ['price', 'fareamount', 'amount', 'farevalue'] and elem.text:
                    try:
                        price = float(elem.text.strip())
                        if price > 0:
                            records.append({'fare_class': 'adult', 'fare_price': price, 'fare_product': 'standard'})
                    except: pass
    except Exception as e:
        print(f"  Error parsing fares XML {xml_file_path}: {e}")
    return records

## PHASE 1: INGESTION

In [17]:
# --- PHASE 1: INGESTION ---
start_time = time.time()
print("=" * 70)
print("PHASE 1: INGESTING 123 TRANSXCHANGE TIMETABLE XML FILES")
print("=" * 70)

timetable_files = sorted(glob.glob(os.path.join(TIMETABLE_XML_DIR, "*.xml")))
assert len(timetable_files) > 0, f"No XML files found in {TIMETABLE_XML_DIR}."

file_paths_rdd = spark.sparkContext.parallelize(timetable_files, numSlices=4)
records_rdd = file_paths_rdd.flatMap(parse_timetable_xml)

schema_timetable = StructType([
    StructField("SourceFile", StringType(), True), StructField("ServiceCode", StringType(), True),
    StructField("LineName", StringType(), True), StructField("VehicleJourneyCode", StringType(), True),
    StructField("JourneyPatternRef", StringType(), True), StructField("DepartureTime", StringType(), True),
    StructField("Sequence", IntegerType(), True)
])

df_timetable = spark.createDataFrame(records_rdd, schema=schema_timetable)
df_timetable.cache()
total_timetable_rows = df_timetable.count()
print(f"✓ Timetable DataFrame created: {total_timetable_rows:,} records")
print(f"⏱ Algorithmic Efficiency - Ingestion Time: {time.time() - start_time:.2f}s")

PHASE 1: INGESTING 123 TRANSXCHANGE TIMETABLE XML FILES


[Stage 54:===========================================>              (3 + 1) / 4]

✓ Timetable DataFrame created: 6,866 records
⏱ Algorithmic Efficiency - Ingestion Time: 8.66s


## PHASE 1b: FARES

In [18]:
# --- PHASE 1b: FARES ---
fare_records = parse_fares_xml(FARES_XML_PATH)
schema_fares = StructType([
    StructField("fare_class", StringType(), True), StructField("fare_price", DoubleType(), True),
    StructField("fare_product", StringType(), True)
])
df_fares = spark.createDataFrame(fare_records, schema=schema_fares)
df_fares.cache()

min_fare = df_fares.agg({"fare_price": "min"}).collect()[0][0] or 1.0
max_fare = df_fares.agg({"fare_price": "max"}).collect()[0][0] or 5.0
avg_fare = df_fares.agg({"fare_price": "avg"}).collect()[0][0] or 2.50

## PHASE 1c: DATA CLEANING

In [19]:
# --- PHASE 1c: DATA CLEANING ---
def drop_empty_columns(df, total_count):
    if total_count == 0: return df
    exprs = []
    for c in df.columns:
        if dict(df.dtypes)[c] == 'string':
            exprs.append((count(when(col(c).isNull() | (col(c) == ""), c)) / total_count).alias(c))
        else:
            exprs.append((count(when(col(c).isNull(), c)) / total_count).alias(c))
    
    null_counts = df.select(exprs).collect()[0].asDict()
    cols_to_keep = [c for c, null_pct in null_counts.items() if null_pct < 1.0]
    return df.select(*cols_to_keep)

df_clean = drop_empty_columns(df_timetable, total_timetable_rows)
df_clean = df_clean.dropDuplicates()
df_clean = df_clean.na.drop(subset=["ServiceCode", "LineName", "DepartureTime"])

## PHASE 1d: DATA AUGMENTATION

In [24]:
# --- PHASE 1d: DATA AUGMENTATION ---
base_features = [c for c in ["SourceFile", "ServiceCode", "LineName", "VehicleJourneyCode", "Sequence", "JourneyPatternRef", "DepartureTime"] if c in df_clean.columns]
df_base = df_clean.select(*base_features)

dates_df = spark.createDataFrame([(i,) for i in range(1, 61)], schema="day_offset INT")

# REQUIREMENT: Broadcast Join Optimization
df_operational = df_base.crossJoin(broadcast(dates_df))

df_operational = df_operational.withColumn("base_date", expr("date_add(current_date(), day_offset)")) \
    .withColumn("time_only", date_format(col("DepartureTime").cast("timestamp"), "HH:mm:ss")) \
    .withColumn("scheduled_departure", to_timestamp(concat(col("base_date"), lit(" "), col("time_only")), "yyyy-MM-dd HH:mm:ss")) \
    .withColumn("hour_of_day", date_format(col("scheduled_departure"), "HH").cast(IntegerType())) \
    .withColumn("day_of_week", date_format(col("scheduled_departure"), "EEEE")) \
    .withColumn("fare_price", least(greatest(lit(avg_fare) + (col("Sequence") * lit(0.10)), lit(min_fare)), lit(max_fare))) \
    .withColumn("delay_minutes", when(
        col("hour_of_day").between(7, 9) | col("hour_of_day").between(16, 18),
        expr("cast((Sequence * 0.2) + (rand() * 15) as int)")
    ).otherwise(
        expr("cast((Sequence * 0.1) + (rand() * 3) as int)")
    ))

df_operational = df_operational.drop("day_offset", "DepartureTime", "base_date", "time_only")

# REQUIREMENT: Repartition (using 4 without key to avoid skew)
df_operational = df_operational.repartition(4)
df_operational.cache()

print(f"\nPartition count: {df_operational.rdd.getNumPartitions()}")
print(f"Partition sizes: {df_operational.rdd.glom().map(len).collect()}")
print(">>> ACTION: Go to Spark UI (http://localhost:4040) and take a screenshot of the Stages tab NOW! <<<")

[Stage 115:==========================================>              (3 + 1) / 4]


Partition count: 4


[Stage 118:============================>                            (2 + 2) / 4]

Partition sizes: [102990, 102990, 102990, 102990]
>>> ACTION: Go to Spark UI (http://localhost:4040) and take a screenshot of the Stages tab NOW! <<<


## PHASE 1e: SAVE TO PARQUET

In [25]:
# --- PHASE 1e: SAVE TO PARQUET ---
df_operational.write.mode("overwrite").parquet(OUTPUT_PARQUET)
df_final = spark.read.parquet(OUTPUT_PARQUET)
df_final.cache()
record_count = df_final.count()

print(f"\n--- BIG DATA THRESHOLD CHECK ---")
print(f"Total Operational Journey Records: {record_count:,}")
if record_count >= 100_000:
    print("✓ Success: Dataset meets the 100,000 record requirement!")

26/08/03 01:28:26 WARN CacheManager: Asked to cache already cached data.        
[Stage 123:>                                                        (0 + 4) / 4]


--- BIG DATA THRESHOLD CHECK ---
Total Operational Journey Records: 411,960
✓ Success: Dataset meets the 100,000 record requirement!


# Phase 2
## EDA, Reliability Metrics & SQLite Storage

In [26]:
import sqlite3
import pandas as pd
from pyspark.sql.functions import desc

df_final.createOrReplaceTempView("bus_operations")

# REQUIREMENT: Full Statistical Measures
print("--- Full Statistical Summary (Mean, Median, Std, Skewness, Kurtosis) ---")
df_final.select(
    mean("delay_minutes").alias("mean"),
    expr("percentile_approx(delay_minutes, 0.5)").alias("median"),
    stddev("delay_minutes").alias("std"),
    skewness("delay_minutes").alias("skewness"),
    kurtosis("delay_minutes").alias("kurtosis")
).show()

# Top 10 Most Delayed Routes
df_top_routes = spark.sql("""
    SELECT LineName, ROUND(AVG(delay_minutes), 2) AS avg_delay_mins, COUNT(*) AS total_trips
    FROM bus_operations GROUP BY LineName ORDER BY avg_delay_mins DESC LIMIT 10
""")

# Average Delay by Hour
df_delay_by_hour = spark.sql("""
    SELECT hour_of_day, ROUND(AVG(delay_minutes), 2) AS avg_delay_mins
    FROM bus_operations GROUP BY hour_of_day ORDER BY hour_of_day ASC
""")

# REQUIREMENT: Reliability & Efficiency Metrics
# 1. Service Reliability (% on-time within +-2 mins)
df_reliability = df_final.withColumn("is_on_time", when(col("delay_minutes") <= 2, 1).otherwise(0))
service_reliability_pct = df_reliability.agg(avg("is_on_time") * 100).collect()[0][0]
print(f"\nService Reliability (On-time %): {service_reliability_pct:.2f}%")

# 2. Travel Time Variability (Coefficient of Variation = std / mean)
df_cvv = df_final.groupBy("LineName").agg(
    (stddev("delay_minutes") / mean("delay_minutes")).alias("travel_time_cv")
).orderBy(desc("travel_time_cv"))
print("\nTop 5 Routes by Travel Time Variability (CV):")
df_cvv.show(5)

--- Full Statistical Summary (Mean, Median, Std, Skewness, Kurtosis) ---
+-----------------+------+----------------+-----------------+------------------+
|             mean|median|             std|         skewness|          kurtosis|
+-----------------+------+----------------+-----------------+------------------+
|3.366681231187494|     2|4.02013937621377|1.400096103464022|0.7072407993853065|
+-----------------+------+----------------+-----------------+------------------+


Service Reliability (On-time %): 67.67%

Top 5 Routes by Travel Time Variability (CV):
+--------+------------------+
|LineName|    travel_time_cv|
+--------+------------------+
|      W7|1.5104183148119663|
|     241|1.3481782455162827|
|     400|1.3280098258176036|
|     781|1.3003999816745409|
|     786|1.2589800184736717|
+--------+------------------+
only showing top 5 rows


In [27]:
# Initialize SQLite Database
conn = sqlite3.connect(DB_PATH)
cursor = conn.cursor()

cursor.execute("""CREATE TABLE IF NOT EXISTS route_delay_metrics (line_name TEXT, avg_delay_mins REAL, total_trips INTEGER, PRIMARY KEY (line_name))""")
cursor.execute("""CREATE TABLE IF NOT EXISTS hourly_delay_metrics (hour_of_day INTEGER, avg_delay_mins REAL, PRIMARY KEY (hour_of_day))""")
cursor.execute("""CREATE TABLE IF NOT EXISTS system_reliability_metrics (metric_name TEXT, metric_value REAL, PRIMARY KEY (metric_name))""")
conn.commit()

pdf_top_routes = df_top_routes.toPandas()
pdf_delay_by_hour = df_delay_by_hour.toPandas()

# Insert using PARAMETERISED QUERIES
print("Inserting metrics into SQLite using parameterised queries...")
for _, row in pdf_top_routes.iterrows():
    cursor.execute("INSERT OR REPLACE INTO route_delay_metrics (line_name, avg_delay_mins, total_trips) VALUES (?, ?, ?)",
                   (row["LineName"], float(row["avg_delay_mins"]), int(row["total_trips"])))

for _, row in pdf_delay_by_hour.iterrows():
    cursor.execute("INSERT OR REPLACE INTO hourly_delay_metrics (hour_of_day, avg_delay_mins) VALUES (?, ?)",
                   (int(row["hour_of_day"]), float(row["avg_delay_mins"])))

# Insert Reliability Metrics
cursor.execute("INSERT OR REPLACE INTO system_reliability_metrics (metric_name, metric_value) VALUES (?, ?)",
               ("Service_Reliability_Pct", float(service_reliability_pct)))

conn.commit()
print("✓ Data successfully saved to SQLite database!")
conn.close()

Inserting metrics into SQLite using parameterised queries...
✓ Data successfully saved to SQLite database!
